# 🌾 Predict Crop Yield in South Sudan
**IndabaX South Sudan 2026 · Scenius Hub**

We'll train a machine learning model to predict how much a farm will harvest (kg per hectare).
Run each cell from top to bottom with `Shift + Enter`.

## 1. Upload the data
Run the cell, click **Choose Files** and pick `data.csv`.

In [ ]:
from google.colab import files
files.upload()

Saving data.csv to data (1).csv


{'data (1).csv': b'region,crop,farm_size_ha,rainfall_mm,avg_temp_c,fertilizer_kg_ha,soil_quality_index,market_access_score,yield_kg_ha,food_security_risk\nCentral Equatoria,Sorghum,2.55,1182.1,28.6,27.8,3.6,4.3,1051.2,Low\nLakes,Cassava,2.0,809.3,29.2,9.1,6.4,8.0,4171.3,Medium\nWarrap,Groundnuts,4.47,839.3,31.2,3.2,4.4,4.2,630.4,Medium\nUnity,Sorghum,1.96,519.5,28.8,8.3,2.5,3.6,551.3,High\nUnity,Sorghum,3.18,,30.0,10.1,,9.1,762.0,Medium\nWestern Bahr el Ghazal,Sorghum,1.03,995.2,27.6,131.0,7.9,4.9,1359.6,Low\nCentral Equatoria,Cassava,0.48,,30.4,24.8,6.3,8.9,5388.6,Low\nWarrap,Maize,2.8,682.3,29.9,3.6,9.1,1.9,1590.7,Low\nWestern Equatoria,Groundnuts,0.47,1387.3,31.2,8.0,5.8,6.1,979.3,Low\nCentral Equatoria,Maize,2.46,876.1,30.8,17.8,9.9,9.6,1974.8,Low\nUpper Nile,Maize,0.7,557.7,30.5,7.0,5.8,5.1,1009.6,Medium\nNorthern Bahr el Ghazal,Sorghum,1.83,887.2,28.8,9.7,5.5,4.4,908.1,Low\nLakes,Cassava,4.52,765.7,29.6,4.8,3.2,2.4,3731.6,Medium\nLakes,Sorghum,2.33,855.4,31.7,2.2,7.5,7.4,865.4,Me

## 2. Load and look at the data

In [ ]:
import pandas as pd

df = pd.read_csv("data.csv").dropna()   # dropna() removes rows with missing values
df.head()

,region,crop,farm_size_ha,rainfall_mm,avg_temp_c,fertilizer_kg_ha,soil_quality_index,market_access_score,yield_kg_ha,food_security_risk
0,Central Equatoria,Sorghum,2.55,1182.1,28.6,27.8,3.6,4.3,1051.2,Low
1,Lakes,Cassava,2.00,809.3,29.2,9.1,6.4,8.0,4171.3,Medium
2,Warrap,Groundnuts,4.47,839.3,31.2,3.2,4.4,4.2,630.4,Medium
3,Unity,Sorghum,1.96,519.5,28.8,8.3,2.5,3.6,551.3,High
5,Western Bahr el Ghazal,Sorghum,1.03,995.2,27.6,131.0,7.9,4.9,1359.6,Low


## 3. Choose the inputs (X) and what to predict (y)

In [ ]:
X = df[["rainfall_mm", "fertilizer_kg_ha", "soil_quality_index"]]   # inputs (features)
y = df["yield_kg_ha"]                                                # answer (label)

In [ ]:
X

,rainfall_mm,fertilizer_kg_ha,soil_quality_index
0,1182.1,27.8,3.6
1,809.3,9.1,6.4
2,839.3,3.2,4.4
3,519.5,8.3,2.5
5,995.2,131.0,7.9
...,...,...,...
895,582.3,8.1,3.3
896,631.9,28.1,8.6
897,893.2,25.5,8.3
898,890.8,64.0,7.1


In [ ]:
y

,yield_kg_ha
0,1051.2
1,4171.3
2,630.4
3,551.3
5,1359.6
...,...
895,917.1
896,1177.8
897,1564.1
898,1876.9


## 4. Split, train and test
The model learns from 80% of farms, then we test it on the 20% it has never seen.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)                     # learning happens here

#error = mean_absolute_error(y_test, model.predict(X_test))
#print("Average error:", round(error), "kg/ha")

RandomForestRegressor(random_state=42)

## 5. 🧪 Predict a new farm
Change the numbers and run again. What happens with less rain? More fertilizer?

In [ ]:
new_farm = pd.DataFrame({
    "rainfall_mm": [550],
    "fertilizer_kg_ha": [10],
    "soil_quality_index": [4.5],
})
print("Predicted yield:", round(model.predict(new_farm)[0]), "kg/ha")

Predicted yield: 772 kg/ha


## 💬 Discuss
Our model doesn't know **which crop** each farm grows, but cassava yields far more per hectare than groundnuts!
How could we make it better? 👇

---
# ⭐ Bonus: improve the model
### A. Add the crop type
Computers need numbers, so `get_dummies` turns each crop into its own 0/1 column.

In [ ]:
X2 = pd.get_dummies(df[["rainfall_mm", "fertilizer_kg_ha", "soil_quality_index", "crop"]])

X_train, X_test, y_train, y_test = train_test_split(X2, y, test_size=0.2, random_state=42)
model2 = RandomForestRegressor(random_state=42)
model2.fit(X_train, y_train)

error2 = mean_absolute_error(y_test, model2.predict(X_test))
print("Before:", round(error), "kg/ha   After adding crop:", round(error2), "kg/ha")

### B. Predict food security risk (Low / Medium / High)
Same steps, but now we predict a **category** instead of a number.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

y_risk = df["food_security_risk"]
X_train, X_test, y_train, y_test = train_test_split(X2, y_risk, test_size=0.2, random_state=42)

risk_model = RandomForestClassifier(random_state=42)
risk_model.fit(X_train, y_train)
print("Accuracy:", f"{accuracy_score(y_test, risk_model.predict(X_test)):.0%}")

## ✅ Done!
Load → choose X and y → split → train → test → predict. That's the workflow you'll use at the hackathon.